# Euclid Q1 NISP

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.utils.data import download_file
from astropy.wcs import WCS
from astropy import units as u

import pyvo as vo
import sep

# Copy-on-write is more performant and avoids unexpected modifications of the original DataFrame.
pd.options.mode.copy_on_write = True

### Download HUDF images in NISP MER Y band from IRSA:
science files, 1 by 1, then look for PSFs

In [ ]:
search_radius = 5 * u.arcmin
coord = SkyCoord.from_name('HUDF')
print(coord)

In [ ]:
irsa_service= vo.dal.sia2.SIA2Service('https://irsa.ipac.caltech.edu/SIA')
image_table = irsa_service.search(pos=(coord, search_radius), collection='euclid_DpdMerBksMosaic',instrument='NISP',data_type='image')
df_im_irsa=image_table.to_table().to_pandas().reset_index()
df_im_euclid=df_im_irsa[ (df_im_irsa['dataproduct_subtype']=='science') & (df_im_irsa['energy_bandpassname']=='Y')].reset_index()
df_im_euclid

In [ ]:
filename = df_im_euclid['access_url'].to_list()[2]
filesize = df_im_euclid['access_estsize'].to_list()[2]/1000000
print(filename,filesize)
fname = download_file(filename, cache=True)
hdu_mer_irsa = fits.open(fname)
print(hdu_mer_irsa.info())
head_mer_irsa = hdu_mer_irsa[0].header

In [ ]:
download_path = '../data/NISP-Y/HUDF/'
hdu_mer_irsa.writeto(os.path.join(download_path, 'MER_image_VIS_2.fits'), overwrite=True)

In [ ]:
#image_table = irsa_service.search(pos=(coord, search_radius),collection='euclid_DpdNirCalibratedFrame',instrument='NISP',data_type='image')
image_table = irsa_service.search(pos=(coord, search_radius), collection='euclid_DpdMerBksMosaic',instrument='NISP',data_type='image')

df_im_irsa=image_table.to_table().to_pandas().reset_index()
df_im_euclid=df_im_irsa[ (df_im_irsa['energy_bandpassname']=='Y')].reset_index()
df_im_euclid

In [ ]:
for i in range(18):
    print(df_im_euclid['access_url'].to_list()[i])


# JWST NIRCAM:

In [ ]:
#https://archive.stsci.edu/hlsp/jades

from astroquery.mast import Observations
f115w_obs = Observations.query_criteria(provenance_name="jades",instrument_name='NIRCAM/IMAGE',filters='F115W')
data_products = Observations.get_product_list(f115w_obs)
Observations.download_products(data_products)

# PSFs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as fits
from matplotlib.colors import LogNorm

### Catalog file

In [ ]:
d = fits.open('../data/NISP-Y/HUDF/PSFs/EUC_MER_CATALOG-PSF-NIR-Y_TILE102044185-B0A5EA_20241021T060047.722344Z_00.00.fits')
d.info()


In [ ]:
d = fits.getdata('../data/NISP-Y/HUDF/PSFs/EUC_MER_CATALOG-PSF-NIR-Y_TILE102044185-B0A5EA_20241021T060047.722344Z_00.00.fits',1)
plt.imshow(d[100:500, 100:500], norm=LogNorm(vmin=0.000001, vmax=0.05),origin='lower')
plt.colorbar()
plt.show()

### Grid file

In [ ]:
d = fits.open('../data/NISP-Y/HUDF/PSFs/EUC_MER_GRID-PSF-NIR-Y_TILE102044185-7B0C0_20241021T003716.460604Z_00.00.fits')
d.info()


In [ ]:
d = fits.getdata('../data/NISP-Y/hudf/PSFs/EUC_MER_GRID-PSF-NIR-Y_TILE102044185-7B0C0_20241021T003716.460604Z_00.00.fits',1)
plt.imshow(d[100:500, 100:500], norm=LogNorm(vmin=0.000001, vmax=0.05),origin='lower')
plt.colorbar()
plt.show()